# eDNA (12S Teleo) dataset check — French Guiana reference library
Upload `Data_12S_Ref_library_Fish_Guiana.csv` and the two Waggoner CSVs (only used in cell 3) to Colab before running.

### CELL 1 — load the reference library

In [2]:
import pandas as pd
ref = pd.read_csv("Data_12S Ref library_Fish Guiana.csv", sep=None, engine="python")
ref["species_name"] = (ref["Genus"].astype(str).str.strip() + " " + ref["Species"].astype(str).str.strip())
ref["seq_full"] = ref["12S sequence"].astype(str).str.lower().str.strip()
ref = ref[ref["12S sequence"].notna()]
print(ref.shape)
print("species:", ref["species_name"].nunique())

(1557, 12)
species: 369


### CELL 2 — cut out the Teleo region (the short piece eDNA studies sequence)

In [3]:
# In this library, the Teleo region sits right after the Teleo forward primer
# (ACACCGCCCGTCACTCT) and runs to the end of the sequence.
PRIMER_F = "acaccgcccgtcactct"
MAX_MISMATCH = 3   # allow small differences, since the primer site varies a bit between species

def find_primer(seq, primer=PRIMER_F, max_mm=MAX_MISMATCH):
    """Return the position right after the best primer match, or None if nothing matches well enough."""
    best_pos, best_mm = None, max_mm + 1
    for i in range(len(seq) - len(primer) + 1):
        mm = sum(a != b for a, b in zip(seq[i:i+len(primer)], primer))
        if mm < best_mm:
            best_pos, best_mm = i + len(primer), mm
    return best_pos

def extract_teleo(seq):
    pos = find_primer(seq)
    return seq[pos:] if pos is not None else None

ref["teleo"] = ref["seq_full"].apply(extract_teleo)
print("sequences where the primer was found:", ref["teleo"].notna().sum(), "/", len(ref))
print(ref["teleo"].str.len().describe())

sequences where the primer was found: 1556 / 1557
count    1556.000000
mean       63.876607
std         4.399526
min        35.000000
25%        63.000000
50%        64.000000
75%        66.000000
max        78.000000
Name: teleo, dtype: float64


### CELL 3 — sanity check: do our cut-out regions match Waggoner's 70-base sequences?

In [6]:
w = pd.concat([pd.read_csv("train_oversampled_t70_noise-0_thresh-2.csv"),
               pd.read_csv("test_t70_noise-0_thresh-2.csv")]).drop_duplicates(subset=["seq", "species"])
w_core = set(w["seq"].str.rstrip("n"))
ours = set(ref["teleo"].dropna())
print("Waggoner sequences we reproduced exactly:", len(w_core & ours), "/", len(w_core))

Waggoner sequences we reproduced exactly: 398 / 465


### CELL 4 — clean species names and drop unusable rows

In [7]:
def is_real_species_name(name):
    parts = name.split()
    if len(parts) != 2:
        return False
    bad = ["sp.", "sp", "aff.", "cf.", "nan"]
    if any(p.lower() in bad for p in parts):
        return False
    if any(ch.isdigit() for ch in name):
        return False
    return True

clean = ref[ref["teleo"].notna() & ref["species_name"].apply(is_real_species_name)].copy()
clean = clean[clean["teleo"].str.len() >= 30]    # drop broken/very short fragments
print("rows kept:", len(clean), "| species:", clean["species_name"].nunique())

rows kept: 1320 | species: 295


### CELL 5 — how many sequences per species?

In [8]:
species_counts = clean["species_name"].value_counts()
print(species_counts.describe())
for t in [1, 2, 3, 5, 8, 10]:
    print(f"species with >= {t} sequences: {(species_counts >= t).sum()}")

# How many species have *distinct* Teleo sequences? (identical short sequences add no new info)
uniq = clean.drop_duplicates(subset=["species_name", "teleo"])
uniq_counts = uniq["species_name"].value_counts()
print("\nafter removing identical duplicates within a species:")
for t in [1, 2, 3, 5]:
    print(f"species with >= {t} distinct sequences: {(uniq_counts >= t).sum()}")

# Species that share the exact same Teleo sequence can't be told apart by any method
shared = uniq.groupby("teleo")["species_name"].nunique()
print("\nTeleo sequences shared by 2+ species:", (shared > 1).sum())

count    295.000000
mean       4.474576
std        4.058903
min        1.000000
25%        1.000000
50%        3.000000
75%        6.000000
max       19.000000
Name: count, dtype: float64
species with >= 1 sequences: 295
species with >= 2 sequences: 202
species with >= 3 sequences: 172
species with >= 5 sequences: 102
species with >= 8 sequences: 58
species with >= 10 sequences: 40

after removing identical duplicates within a species:
species with >= 1 distinct sequences: 295
species with >= 2 distinct sequences: 120
species with >= 3 distinct sequences: 58
species with >= 5 distinct sequences: 10

Teleo sequences shared by 2+ species: 6


### CELL 6 — check against the fish tree (same tree file as your BOLD notebook)

In [9]:
# CELL 6 — check against the FULL fish tree (all-taxon, ~31k species)
!pip -q install dendropy
!wget -q -O actinopt_full.trees.xz "https://fishtreeoflife.org/downloads/actinopt_full.trees.xz"
# the file holds 100 versions of the tree (one per line) — keep just the first one
!xz -dc actinopt_full.trees.xz | head -n 1 > fish_tree_full.tre

import dendropy
tree = dendropy.Tree.get(path="fish_tree_full.tre", schema="newick",
                         preserve_underscores=True)
tree_species = set(leaf.taxon.label.replace("_", " ").strip("'")
                   for leaf in tree.leaf_node_iter())
print("species in full tree:", len(tree_species))

all_species = set(species_counts.index)
in_tree = all_species & tree_species
print(f"our species in tree: {len(in_tree)} / {len(all_species)} ({100*len(in_tree)/len(all_species):.1f}%)")
print("missing (first 20):", sorted(all_species - tree_species)[:20])

in_tree_counts = species_counts[list(in_tree)]
for t in [2, 3, 5]:
    print(f"in-tree species with >= {t} sequences: {(in_tree_counts >= t).sum()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.4/505.4 kB 11.3 MB/s eta 0:00:00
species in full tree: 31516
our species in tree: 271 / 295 (91.9%)
missing (first 20): ['Bryconops florenceae', 'Bunocephalus aloikae', 'Curculionichthys monolechis', 'Distocyclus guchereauae', 'Gephyromochlus leopardus', 'Guyanancistrus brevispinis', 'Guyanancistrus longispinis', 'Guyanancistrus megastictus', 'Guyanancistrus niger', 'Lepidosiren paradoxa', 'Lugubria johanna', 'Mastiglanis durantoni', 'Myloplus rhomboidalis', 'Myloplus ternetzi', 'Paralithoxus boujardi', 'Paralithoxus planquettei', 'Paralithoxus stocki', 'Phenacorhamdia tenuis', 'Potamotrygon marinae', 'Potamotrygon orbignyi']
in-tree species with >= 2 sequences: 190
in-tree species with >= 3 sequences: 163
in-tree species with >= 5 sequences: 95


### CELL 7 — save the clean dataset for the next stage

In [12]:
clean["in_tree"] = clean["species_name"].isin(tree_species)

clean[["Specimen identification code", "Order", "Family", "Genus",
       "species_name", "teleo", "in_tree"]] \
    .to_csv("guiana_teleo_clean.csv", index=False)

print("saved guiana_teleo_clean.csv")
print("rows in tree:", clean["in_tree"].sum(), "/", len(clean))

saved guiana_teleo_clean.csv
rows in tree: 1231 / 1320


In [13]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/phylo-novelty-edna/data"
!cp guiana_teleo_clean.csv fish_tree_full.tre "/content/drive/MyDrive/phylo-novelty-edna/data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
